# 03 — Producción: perfilado de la producción mensual por pozo (2024–2026)

**Objetivo:** conocer la estructura y la calidad de los archivos de producción **antes de limpiar nada**.
Este notebook solo *mide*: no modifica ni guarda datos.

| Dato | Valor |
| --- | --- |
| Archivos de origen | `data/raw/produccin-de-pozos-de-gas-y-petrleo-2024.csv`, `-2025.csv` y `-2026.csv` |
| Fuente | Secretaría de Energía — ver `data/raw/README.md` |
| Qué contiene | Producción mensual de cada pozo: petróleo, gas, agua, días de producción, empresa, formación |
| Período | Enero 2024 al último mes publicado de 2026 |
| Política aplicada | `POLITICA_CALIDAD_DATOS.md` — paso 1: **Perfilar** |
| Etapa anterior | `02_pozos_limpieza.ipynb` — genera el maestro de pozos limpio, que se usa acá para verificar la trazabilidad |
| Etapa siguiente | `04_produccion_limpieza.ipynb` — aplica las decisiones sobre estos hallazgos |

**Unidades** (documentadas por la fuente): petróleo en **m³**, gas en **miles de m³**.

**Decisiones de alcance** (tomadas antes de este notebook):
- El período es **2024–2026**: tres años alcanzan para comparar la producción reciente y mantienen el volumen manejable.
- No se usa `produccin-de-pozos-de-gas-y-petrleo-no-convencional.csv`: sus filas de 2024–2026 ya están en los archivos anuales, y usarlo duplicaría datos.
- **Salvedad de oportunidad:** los archivos 2024 y 2025 se descargaron el 13/10/2026; el de 2026, el 23/09/2026.
  Si la Secretaría corrigió algún mes de 2026 entre esas fechas, la corrección no está en este análisis.

**Trazabilidad:** cada problema detectado recibe un código **H-xx**. La numeración sigue la del notebook 01 (H-01 a H-27),
para que el proyecto tenga un solo registro de hallazgos. Los de producción empiezan en **H-28**.

## 1. Preparar el entorno

- **pandas** trabaja con tablas.
- **hashlib** calcula la huella SHA-256 de un archivo (su "huella digital"): permite probar al final que los raw no cambiaron.
- **os** permite consultar el tamaño de los archivos.

Mostramos la versión de pandas porque forma parte de la reproducibilidad.

In [ ]:
import hashlib
import os

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
print("Versión de pandas:", pd.__version__)

## 2. Ubicar los archivos y tomar su huella digital

Definimos las rutas de los tres archivos anuales y calculamos la **huella SHA-256** de cada uno **antes de leerlos**.
Al final del notebook la volvemos a calcular: si coincide, queda probado que el notebook no modificó los originales.

La función lee el archivo en bloques de 1 MB, porque los archivos pesan más de 200 MB y no conviene cargarlos enteros solo para esto.

También ubicamos el **maestro de pozos limpio** (`data/clean`), que usamos en la sección 9 para la trazabilidad.

In [ ]:
CARPETA_RAW = "../data/raw"
ARCHIVOS = {
    2024: "produccin-de-pozos-de-gas-y-petrleo-2024.csv",
    2025: "produccin-de-pozos-de-gas-y-petrleo-2025.csv",
    2026: "produccin-de-pozos-de-gas-y-petrleo-2026.csv",
}
RUTA_MAESTRO = "../data/clean/capitulo-iv-pozos_limpio.csv"


def huella_sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()


huellas_inicio = {}
for anio, nombre in ARCHIVOS.items():
    ruta = os.path.join(CARPETA_RAW, nombre)
    huellas_inicio[nombre] = huella_sha256(ruta)
    print(f"{nombre}  |  {os.path.getsize(ruta) / 1e6:,.1f} MB  |  SHA-256: {huellas_inicio[nombre]}")

## 3. Estructura: ¿los tres archivos tienen las mismas columnas?

Para unir los tres años en una sola tabla, tienen que tener las mismas columnas.
Leemos **solo el encabezado** de cada archivo (`nrows=0`: cero filas), que es instantáneo, y comparamos.

Si una columna está en un archivo y no en otro, es un hallazgo de **consistencia** entre archivos de la misma fuente.

In [ ]:
columnas_por_archivo = {
    nombre: list(pd.read_csv(os.path.join(CARPETA_RAW, nombre), nrows=0).columns)
    for nombre in ARCHIVOS.values()
}

todas = sorted(set().union(*columnas_por_archivo.values()))
presencia = pd.DataFrame(
    {nombre: [col in cols for col in todas] for nombre, cols in columnas_por_archivo.items()},
    index=todas,
)
print("Columnas por archivo:", {n: len(c) for n, c in columnas_por_archivo.items()})
print("\nColumnas que NO están en todos los archivos:")
presencia[~presencia.all(axis=1)]

## 4. Cargar solo las columnas necesarias

Los tres archivos suman más de 800 MB. Leer las 39 columnas puede agotar la memoria de la computadora,
así que leemos **solo las que necesitamos** para el perfilado y los gráficos del EDA (`usecols`).
Las demás (agua, inyección, usuario de carga, etc.) quedan en el raw, intactas, por si se necesitan después.

| Grupo | Columnas |
| --- | --- |
| Identificación y tiempo | `idpozo`, `sigla`, `anio`, `mes`, `idempresa`, `empresa` |
| Producción | `prod_pet` (m³), `prod_gas` (miles de m³), `tef` (tiempo efectivo de producción) |
| Clasificación | `tipo_de_recurso`, `sub_tipo_recurso`, `formacion`, `formprod`, `cuenca`, `provincia`, `areayacimiento`, `tipoestado`, `tipopozo` |
| Control | `rectificado`, `habilitado`, `fechaingreso`, `fecha_data` |

Agregamos una columna **`archivo_origen`** con el nombre del archivo de cada fila.
Es **linaje**: cualquier fila se puede rastrear hasta su archivo raw, incluso después de unir los tres años.

`pd.concat` apila las tres tablas una debajo de la otra.

In [ ]:
COLUMNAS = [
    "idpozo", "sigla", "anio", "mes", "idempresa", "empresa",
    "prod_pet", "prod_gas", "tef",
    "tipo_de_recurso", "sub_tipo_recurso", "formacion", "formprod", "cuenca", "provincia",
    "areayacimiento", "tipoestado", "tipopozo",
    "rectificado", "habilitado", "fechaingreso", "fecha_data",
]

partes = []
for anio, nombre in ARCHIVOS.items():
    parte = pd.read_csv(os.path.join(CARPETA_RAW, nombre), usecols=COLUMNAS, low_memory=False)
    parte["archivo_origen"] = nombre
    partes.append(parte)
    print(f"{nombre}: {len(parte):,} filas")

df = pd.concat(partes, ignore_index=True)
del partes  # libera memoria: ya no hacen falta las tablas sueltas

print(f"\nTotal: {len(df):,} filas  |  {df.shape[1]} columnas  |  "
      f"memoria: {df.memory_usage(deep=True).sum() / 1e6:,.0f} MB")

## 5. Primer vistazo: tipos de datos y ejemplo de filas

`info()` muestra el **tipo de dato** de cada columna (`int64` = entero, `float64` = decimal, `str` = texto).
Un tipo inesperado es una pista: por ejemplo, si `prod_pet` saliera como texto, habría algún valor no numérico.

Después miramos 3 filas transpuestas (`.T`), para ver las columnas como filas.

In [ ]:
df.info()

In [ ]:
df.head(3).T

## 6. Completitud: ¿falta algún dato?

Contamos por columna:
- **Nulos**: celdas vacías.
- **"No informado"**: nulos ocultos, como en el maestro de pozos (H-03 a H-06). `isna()` no los detecta.

**Cero no es nulo** (criterio de oro de la política): un pozo con producción 0 puede estar parado, y es un dato real.
Por eso los ceros de producción se cuentan **aparte** y no como faltantes.

In [ ]:
completitud = pd.DataFrame({
    "nulos": df.isna().sum(),
    "no_informado": (df == "No informado").sum(),
})
completitud["faltantes_total"] = completitud["nulos"] + completitud["no_informado"]
completitud["pct_faltante"] = (completitud["faltantes_total"] / len(df) * 100).round(2)

completitud[completitud["faltantes_total"] > 0].sort_values("pct_faltante", ascending=False)

Ahora los **ceros**: no son faltantes, pero conviene conocerlos.
Para el histograma en escala logarítmica del EDA (gráfico 3) importan, porque el logaritmo de 0 no existe.

In [ ]:
ceros = pd.DataFrame({
    "filas": [
        (df["prod_pet"] == 0).sum(),
        (df["prod_gas"] == 0).sum(),
        ((df["prod_pet"] == 0) & (df["prod_gas"] == 0)).sum(),
    ]
}, index=["prod_pet = 0", "prod_gas = 0", "petróleo y gas = 0"])
ceros["pct"] = (ceros["filas"] / len(df) * 100).round(1)
ceros

## 7. Unicidad: ¿hay registros repetidos?

La producción se informa **una vez por pozo y por mes**. Entonces la clave esperada es **`idpozo` + `anio` + `mes`**.

Revisamos:
1. **Filas completas duplicadas** (sin contar `archivo_origen`): la misma fila exacta dos veces.
2. **Clave repetida**: el mismo pozo y mes más de una vez, aunque el resto de la fila cambie.
3. Si la clave se repite, **¿dentro del mismo archivo o entre archivos distintos?** Entre archivos indicaría, por ejemplo, que un mes de 2025 también está en el archivo 2026.

`keep=False` marca **todas** las filas del grupo repetido, no solo la segunda, para poder verlas juntas.

In [ ]:
CLAVE = ["idpozo", "anio", "mes"]
columnas_dato = [c for c in df.columns if c != "archivo_origen"]

filas_duplicadas = df.duplicated(subset=columnas_dato)
clave_repetida = df.duplicated(subset=CLAVE, keep=False)
archivos_por_clave = df[clave_repetida].groupby(CLAVE)["archivo_origen"].nunique()

print(f"Filas completas duplicadas:            {filas_duplicadas.sum():,}")
print(f"Filas con clave pozo+mes repetida:     {clave_repetida.sum():,}")
print(f"  - grupos repetidos dentro de 1 archivo: {(archivos_por_clave == 1).sum():,}")
print(f"  - grupos repetidos entre archivos:      {(archivos_por_clave > 1).sum():,}")

Algunos ejemplos de clave repetida, ordenados para ver juntas las filas del mismo pozo y mes
(si no hay repetidos, la tabla sale vacía).

In [ ]:
df[clave_repetida].sort_values(CLAVE)[CLAVE + ["prod_pet", "prod_gas", "tef", "rectificado", "archivo_origen"]].head(10)

## 8. Validez: ¿los valores respetan su formato y rango?

| Regla | Por qué |
| --- | --- |
| `anio` coincide con el año del archivo | Cada archivo anual debería traer solo su año |
| `mes` entre 1 y 12 | Un mes 0 o 13 es imposible |
| `prod_pet` y `prod_gas` no negativos | No se puede producir un volumen negativo |
| `tef` entre 0 y los días del mes | Un pozo no puede producir más días de los que tiene el mes. **Supuesto:** `tef` está en días; la fuente no lo documenta (a confirmar) |

Para saber cuántos días tiene cada mes, armamos una fecha con el día 1 de cada `anio`-`mes`
y usamos `days_in_month` (contempla los años bisiestos). `errors="coerce"` deja vacía la fecha si el mes es inválido, en lugar de frenar con un error.

In [ ]:
anio_archivo = df["archivo_origen"].map({nombre: anio for anio, nombre in ARCHIVOS.items()})

fecha_mes = pd.to_datetime(
    df["anio"].astype("Int64").astype(str) + "-" + df["mes"].astype("Int64").astype(str) + "-01",
    errors="coerce",
)
dias_del_mes = fecha_mes.dt.days_in_month

validez = pd.Series({
    "anio distinto al del archivo": (df["anio"] != anio_archivo).sum(),
    "mes fuera de 1 a 12": (~df["mes"].between(1, 12)).sum(),
    "prod_pet negativa": (df["prod_pet"] < 0).sum(),
    "prod_gas negativa": (df["prod_gas"] < 0).sum(),
    "tef negativo": (df["tef"] < 0).sum(),
    "tef mayor que los días del mes": (df["tef"] > dias_del_mes).sum(),
}, name="filas")
validez.to_frame().assign(pct=(validez / len(df) * 100).round(3))

Rango observado de las columnas numéricas, para tener una idea de los valores.
Si `tef` tuviera máximos muy por encima de 31, el supuesto de que está en días sería falso (se revisa en las conclusiones).

In [ ]:
df[["anio", "mes", "prod_pet", "prod_gas", "tef"]].describe(percentiles=[0.5, 0.99]).round(2)

## 9. Consistencia y trazabilidad

### 9.1 Valores de las clasificaciones

Listamos los valores de las columnas categóricas que usan los gráficos del EDA.
Buscamos variantes de escritura (mayúsculas, espacios) o valores raros.
`dropna=False` incluye los nulos en el conteo.

In [ ]:
for columna in ["tipo_de_recurso", "sub_tipo_recurso", "cuenca", "provincia", "rectificado", "habilitado"]:
    print(f"--- {columna} ---")
    print(df[columna].value_counts(dropna=False).to_string(), "\n")

### 9.2 ¿Un pozo cambia de clasificación o de empresa en el período?

Para cada `idpozo` contamos cuántos valores distintos de `tipo_de_recurso` y de `empresa` tiene en los tres años (`nunique`).
- **Más de un tipo de recurso:** sospechoso; un pozo no deja de ser convencional. "No informado" no cuenta como un tipo distinto (ya se mide en completitud).
- **Más de una empresa:** puede ser real (venta del área o cambio de operador). Se registra **para investigar**, no como error.

Las cifras se informan en filas de producción afectadas, como pide la política.

In [ ]:
tipo_informado = df["tipo_de_recurso"].where(df["tipo_de_recurso"] != "No informado")
tipos_por_pozo = tipo_informado.groupby(df["idpozo"]).nunique()
empresas_por_pozo = df.groupby("idpozo")["empresa"].nunique()

pozos_varios_tipos = tipos_por_pozo[tipos_por_pozo > 1].index
pozos_varias_empresas = empresas_por_pozo[empresas_por_pozo > 1].index

filas_varios_tipos = df["idpozo"].isin(pozos_varios_tipos)
filas_varias_empresas = df["idpozo"].isin(pozos_varias_empresas)

print(f"Pozos (idpozo) en el período:            {df['idpozo'].nunique():,}")
print(f"Pozos con más de un tipo_de_recurso:     {len(pozos_varios_tipos):,}  ({filas_varios_tipos.sum():,} filas)")
print(f"Pozos con más de una empresa:            {len(pozos_varias_empresas):,}  ({filas_varias_empresas.sum():,} filas)")

### 9.3 `formacion` y `formprod`: ¿dicen lo mismo?

Hay dos columnas de formación. Por el nombre, `formprod` parece un **código** de la formación productiva y `formacion`, su **nombre** (a confirmar).
Si es así, cada código debería corresponder a un solo nombre. Contamos los códigos con más de un nombre.

In [ ]:
nombres_por_codigo = df.groupby("formprod")["formacion"].nunique()
print(f"Códigos formprod: {len(nombres_por_codigo):,}  |  con más de un nombre de formación: {(nombres_por_codigo > 1).sum():,}")

df.groupby(["formprod", "formacion"], dropna=False).size().sort_values(ascending=False).head(15).to_frame("filas")

### 9.4 ¿Cómo aparece Vaca Muerta?

El gráfico 4 del EDA filtra los pozos de Vaca Muerta. Antes de decidir el filtro, vemos **cómo está escrita**
en las dos columnas de formación. `str.contains("vaca", case=False)` busca el texto "vaca" sin importar mayúsculas.

In [ ]:
es_vaca = df["formacion"].str.contains("vaca", case=False, na=False)
df[es_vaca].groupby(["formprod", "formacion", "tipo_de_recurso"], dropna=False).size().to_frame("filas")

### 9.5 Trazabilidad: ¿todos los pozos existen en el maestro limpio?

Todo resultado debe poder rastrearse hasta su origen. Si un `idpozo` de producción **no está en el maestro de pozos**,
no podemos conocer su ficha (profundidad, coordenadas, clasificación). Es un problema de **integridad referencial**.

También comparamos el tipo de recurso: `tipo_de_recurso` (producción) contra `tipo_recurso` (maestro limpio).
Solo se comparan los pozos que tienen el dato en los dos archivos.
En el maestro, los "No informado" ya son nulos (regla R-02 del notebook 02), así que acá también excluimos los "No informado".

In [ ]:
maestro = pd.read_csv(RUTA_MAESTRO, usecols=["idpozo", "tipo_recurso"], low_memory=False)

no_en_maestro = ~df["idpozo"].isin(maestro["idpozo"])
print(f"Filas de producción con idpozo que no está en el maestro: {no_en_maestro.sum():,}  "
      f"({df.loc[no_en_maestro, 'idpozo'].nunique():,} pozos)")

comparacion = df[["idpozo", "tipo_de_recurso"]].merge(maestro, on="idpozo", how="inner")
con_dato = comparacion["tipo_de_recurso"].notna() & (comparacion["tipo_de_recurso"] != "No informado") \
    & comparacion["tipo_recurso"].notna()
tipo_distinto = con_dato & (comparacion["tipo_de_recurso"] != comparacion["tipo_recurso"])
print(f"Filas con tipo de recurso distinto al del maestro:        {tipo_distinto.sum():,}  "
      f"({comparacion.loc[tipo_distinto, 'idpozo'].nunique():,} pozos)")

comparacion[tipo_distinto].groupby(["tipo_de_recurso", "tipo_recurso"]).size().to_frame("filas")

## 10. Exactitud y oportunidad

### 10.1 Valores extremos

Un outlier **no se borra automáticamente**: primero se investiga (criterio de oro).
Mostramos los 10 meses de mayor producción de petróleo y de gas, con el pozo y su clasificación,
para juzgar si son creíbles (por ejemplo, un pozo de Vaca Muerta puede producir mucho más que uno convencional).

In [ ]:
ver = ["idpozo", "sigla", "anio", "mes", "prod_pet", "prod_gas", "tef", "tipo_de_recurso", "formacion", "empresa"]
print("Top 10 meses por petróleo (m³):")
display(df.nlargest(10, "prod_pet")[ver])
print("Top 10 meses por gas (miles de m³):")
display(df.nlargest(10, "prod_gas")[ver])

### 10.2 ¿Hasta qué mes llegan los datos?

Contamos las filas por mes. Si los **últimos meses** tienen muchas menos filas que los anteriores,
probablemente las empresas todavía no terminaron de cargar su producción (demora de carga).
Usar esos meses en el EDA mostraría una **caída falsa** de la producción.

`fecha_data` parece indicar la fecha de actualización del dato (a confirmar): mostramos sus valores.

In [ ]:
filas_por_mes = df.groupby(["anio", "mes"]).agg(
    filas=("idpozo", "size"),
    pozos=("idpozo", "nunique"),
    prod_pet_m3=("prod_pet", "sum"),
    prod_gas_miles_m3=("prod_gas", "sum"),
)
print("Últimos 6 meses:")
display(filas_por_mes.tail(6))
print("Valores de fecha_data:")
df["fecha_data"].value_counts().sort_index().tail(10)

## 11. Registro de hallazgos

Cada control recibe un **código H-xx** y se mide en filas y porcentaje.
**Filas = 0 significa que el control se pasó sin problemas** (no hay hallazgo), pero queda en el registro como evidencia de que se revisó.

Se calcula con código para que, si los raw se actualizan, el registro se actualice solo.
El **estado** de todos es *Pendiente*: en este notebook solo se diagnostica; las decisiones se toman en `04_produccion_limpieza.ipynb`.

In [ ]:
total = len(df)
archivos_sin_alguna_columna = [n for n in presencia.columns if not presencia[n].all()]
filas_archivos_incompletos = df["archivo_origen"].isin(archivos_sin_alguna_columna).sum()

controles = [
    ("H-28", "Consistencia", "Archivos anuales con columnas distintas (filas de los archivos a los que les falta alguna)", filas_archivos_incompletos),
    ("H-29", "Completitud", "prod_pet vacía", df["prod_pet"].isna().sum()),
    ("H-30", "Completitud", "prod_gas vacía", df["prod_gas"].isna().sum()),
    ("H-31", "Completitud", "tipo_de_recurso vacío o 'No informado'",
     (df["tipo_de_recurso"].isna() | (df["tipo_de_recurso"] == "No informado")).sum()),
    ("H-32", "Unicidad", "Filas completas duplicadas", filas_duplicadas.sum()),
    ("H-33", "Unicidad", "Clave idpozo + anio + mes repetida", clave_repetida.sum()),
    ("H-34", "Validez", "anio distinto al del archivo", validez["anio distinto al del archivo"]),
    ("H-35", "Validez", "mes fuera de 1 a 12", validez["mes fuera de 1 a 12"]),
    ("H-36", "Validez", "prod_pet negativa", validez["prod_pet negativa"]),
    ("H-37", "Validez", "prod_gas negativa", validez["prod_gas negativa"]),
    ("H-38", "Validez", "tef negativo o mayor que los días del mes",
     validez["tef negativo"] + validez["tef mayor que los días del mes"]),
    ("H-39", "Consistencia", "Pozo con más de un tipo_de_recurso en el período", filas_varios_tipos.sum()),
    ("H-40", "Consistencia", "Pozo con más de una empresa en el período (a investigar: puede ser real)", filas_varias_empresas.sum()),
    ("H-41", "Consistencia", "Código formprod con más de un nombre de formación",
     df["formprod"].isin(nombres_por_codigo[nombres_por_codigo > 1].index).sum()),
    ("H-42", "Consistencia", "idpozo que no existe en el maestro de pozos limpio", no_en_maestro.sum()),
    ("H-43", "Consistencia", "tipo_de_recurso distinto del tipo_recurso del maestro", tipo_distinto.sum()),
]

registro = pd.DataFrame(controles, columns=["codigo", "dimension", "control", "filas"])
registro["pct"] = (registro["filas"] / total * 100).round(2)
registro["estado"] = registro["filas"].map(lambda n: "Pendiente" if n > 0 else "Sin hallazgo")
registro

## 12. Verificación final: los raw no cambiaron

Recalculamos la huella SHA-256 de cada archivo raw y la comparamos con la del inicio (sección 2).
`assert` frena el notebook con un error si alguna no coincide.

In [ ]:
for nombre, huella in huellas_inicio.items():
    assert huella_sha256(os.path.join(CARPETA_RAW, nombre)) == huella, f"¡{nombre} cambió!"
print("Los 3 archivos raw están sin modificar: OK (misma huella que al inicio)")

## 13. Conclusiones y decisiones pendientes

*Se completa después de la primera ejecución con los datos reales, a partir del registro de hallazgos.*